# "Scratch" Transformer

In [2]:
import tensorflow as tf
from tensorflow.keras import layers

class SimpleSelfAttention(layers.Layer):
    def __init__(self, d_model):
        super(SimpleSelfAttention, self).__init__()
        self.d_model = d_model
        
        # 1. Define the linear transforms for Q, K, and V
        self.query_dense = layers.Dense(d_model)
        self.key_dense = layers.Dense(d_model)
        self.value_dense = layers.Dense(d_model)

    def call(self, x):
        # x shape: (batch_size, seq_len, d_model)
        
        # 2. Create Q, K, and V
        q = self.query_dense(x)
        k = self.key_dense(x)
        v = self.value_dense(x)

        # 3. Calculate Attention Scores (Q * K_transposed)
        # This tells us how much every word relates to every other word
        matmul_qk = tf.matmul(q, k, transpose_b=True)

        # 4. Scale the scores 
        # (This prevents the numbers from getting too big for softmax)
        dk = tf.cast(tf.shape(k)[-1], tf.float32)
        scaled_attention_logits = matmul_qk / tf.math.sqrt(dk)

        # 5. Softmax to get probabilities (0 to 1)
        attention_weights = tf.nn.softmax(scaled_attention_logits, axis=-1)

        # 6. Multiply weights by Values
        output = tf.matmul(attention_weights, v)
        
        return output

2026-01-13 15:33:52.111826: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-01-13 15:33:53.314124: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-01-13 15:33:55.387679: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [3]:
class TransformerBlock(layers.Layer):
    def __init__(self, d_model, num_heads, ff_dim, rate=0.1):
        super(TransformerBlock, self).__init__()
        # MultiHeadAttention is built into Keras/TensorFlow!
        self.att = layers.MultiHeadAttention(num_heads=num_heads, key_dim=d_model)
        self.ffn = tf.keras.Sequential([
            layers.Dense(ff_dim, activation="relu"), 
            layers.Dense(d_model)
        ])
        self.layernorm1 = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = layers.Dropout(rate)
        self.dropout2 = layers.Dropout(rate)

    def call(self, inputs, training=True):
        # 1. Self-Attention + Residual Connection
        attn_output = self.att(inputs, inputs)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)
        
        # 2. Feed-Forward + Residual Connection
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)

In [5]:
# Shape: (batch_size=3, seq_len=10, d_model=64)
data = tf.random.normal((3, 10, 64))

attention_layer = SimpleSelfAttention(d_model=64)
attention_layer.call(data)

<tf.Tensor: shape=(3, 10, 64), dtype=float32, numpy=
array([[[ 0.6907876 ,  0.07966159,  0.4465222 , ...,  1.3900014 ,
          0.2527717 ,  0.477937  ],
        [ 0.61425793,  0.24539539,  0.846092  , ...,  0.81889236,
          0.16726176,  0.30463594],
        [ 0.4190648 , -0.269435  , -0.01201021, ...,  0.32331145,
          0.37791616,  0.44492415],
        ...,
        [ 0.17070897,  0.3741138 , -0.01156764, ...,  0.6929122 ,
         -0.03433332, -0.37279004],
        [ 0.10807559,  0.17905451, -0.2652881 , ...,  0.61922437,
          0.08046357,  0.07290255],
        [ 0.407177  ,  0.25522923, -0.32432026, ...,  0.72205466,
         -0.22106412,  0.07359391]],

       [[ 0.15049094,  0.4582107 ,  0.01885849, ..., -0.18600926,
         -0.11189806,  0.2722652 ],
        [ 0.03318235,  0.8099348 , -0.1103515 , ...,  0.1313724 ,
         -0.27152228, -0.35954037],
        [-0.13617055,  1.2543734 , -0.06316388, ...,  0.17478316,
         -0.22596765, -0.31384224],
        ...,
 

In [6]:
class MyTransformerModel(tf.keras.Model):
    def __init__(self, vocab_size, d_model, num_heads, ff_dim, num_classes):
        super().__init__()
        # 1. Represent words as vectors
        self.embedding = layers.Embedding(vocab_size, d_model)
        # 2. Add our custom block
        self.transformer_block = TransformerBlock(d_model, num_heads, ff_dim)
        # 3. Reduce the sequence to a single vector (Global Average Pooling)
        self.pooling = layers.GlobalAveragePooling1D()
        # 4. Final classification layer
        self.classifier = layers.Dense(num_classes)

    def call(self, inputs):
        x = self.embedding(inputs)
        x = self.transformer_block(x)
        x = self.pooling(x)
        return self.classifier(x)

In [ ]:
# 1. Setup the specs
VOCAB_SIZE = 5000  # Number of words in our "dictionary"
D_MODEL = 64       # Size of our word vectors
NUM_HEADS = 4
FF_DIM = 128
NUM_CLASSES = 2    # Positive vs Negative

# 2. Instantiate the model
scratch_model = MyTransformerModel(VOCAB_SIZE, D_MODEL, NUM_HEADS, FF_DIM, NUM_CLASSES)

# 3. Create dummy input: 3 sentences, 10 words each
# We use random integers between 0 and 4999
dummy_ids = tf.random.uniform((3, 10), minval=0, maxval=VOCAB_SIZE, dtype=tf.int32)

# 4. Run the model
predictions = scratch_model(dummy_ids)

print(f"Output Shape: {predictions.shape}") # (3, 2)